# Prepare Atomic Files

Loads the filtered CSV from `process-jsonl.ipynb` and converts it into RecBole Atomic Files.
Run `process-jsonl.ipynb` first to generate `data/processed/reviews.csv.gz`.

In [55]:
from pathlib import Path
import pandas as pd

In [56]:
# --- Config ---
TARGET_CATEGORY = "Beauty_and_Personal_Care"
DATA_DIR: str = "../data"

# Data split ratios
TRAIN_RATIO: float = 0.8
VALID_RATIO: float = 0.1

# Cold vs. warm users
WARM_USER_MIN_REVIEWS: int = 5

# Sequential recommendation config
MAX_ITEM_LIST_LENGTH: int = 50

## Load reviews

In [57]:
df_reviews = pd.read_csv(f"{DATA_DIR}/reviews.csv")
display(df_reviews.sample(10))
df_reviews.info()

,user_id,parent_asin,rating,timestamp,category
41787,AHR2MK5MEXDXYWRC6US5WIYX5MXA,B08QTSW4TH,5.0,1622754548754,Beauty_and_Personal_Care
83616,AHISU6WTZZJNNVYK5A5UITDIWCNQ,B0BG87PDSW,5.0,1668896769084,Beauty_and_Personal_Care
85660,AG4ARTYMP2Q7FF6A2ECB4WMK67WA,B0CB1XF8JS,5.0,1620553170466,Beauty_and_Personal_Care
753,AHKIMFUXMLOUN7SBXHEDD2K2AN7Q,B0C8SLP5WX,4.0,1643994357711,Beauty_and_Personal_Care
180709,AHJQK7K4FA7WWOY4DNPF7LCZZBTQ,B0BVJ8JHP4,5.0,1618971960238,Clothing_Shoes_and_Jewelry
32788,AHO5M4XVPUJ5PJSCZFBJEAUTLUIA,B0BNQDP34Y,5.0,1582004276050,Beauty_and_Personal_Care
173555,AH5HSMUMRJRDWBPFDQ6G3EAAOSFQ,B0BM4XPC3S,5.0,1656622893411,Clothing_Shoes_and_Jewelry
101683,AGOWLR52GJKEQ6ARUX5V6TJP7QLA,B0885XVRGH,4.0,1616975873253,Clothing_Shoes_and_Jewelry
145879,AEBXJRP4COCKP22LPKUDVCQ7JKVQ,B01MZ8ZV5M,5.0,1583960590752,Clothing_Shoes_and_Jewelry
116669,AEZUH6OWMCICUJ2MWPBLFSZTSTUA,B0749HPQQD,5.0,1603340242254,Clothing_Shoes_and_Jewelry


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   user_id      200000 non-null  object 
 1   parent_asin  200000 non-null  object 
 2   rating       200000 non-null  float64
 3   timestamp    200000 non-null  int64  
 4   category     200000 non-null  object 
dtypes: float64(1), int64(1), object(3)
memory usage: 7.6+ MB


## Load items

In [58]:
df_items = pd.read_csv(f"{DATA_DIR}/items.csv")
display(df_items.sample(10))
df_items.info()

,parent_asin,title,price,store,category
4972,B08BRDXT4G,Oneleaf Styling Hair Comb 10PCS Hair Stylists ...,6.99,oneleaf,Beauty_and_Personal_Care
56911,B096YX8YVM,Bali Women's Flower Underwire Bra,18.99,Bali,Clothing_Shoes_and_Jewelry
63415,B0BCX37QLP,PKONEWNO Leather Backpack Purse for Women Fash...,22.99,PKONEWNO,Clothing_Shoes_and_Jewelry
78080,B00TPPD3WS,Ladies Floral Chakras Burnout V Hoodie,42.0,Yoga Clothing For You,Clothing_Shoes_and_Jewelry
40924,B08Y7XHPV7,"Warmstore Small Makeup Bags, 2 Pack Travel Cos...",NaN,Warmstore,Beauty_and_Personal_Care
76551,B0872D3V3Z,Under Armour Boys' Prototype 2.0 Shorts,17.65,Under Armour,Clothing_Shoes_and_Jewelry
73559,B07T1DHPB5,Timberland Women's Leather RFID Flap Wallet Cl...,29.99,Timberland,Clothing_Shoes_and_Jewelry
52546,B08592B34W,Emma Hardie Vitamin C Intense Daily Serum 1 Oz,55.99,NaN,Beauty_and_Personal_Care
71674,B07JGNWT7F,Mippo Long Sleeve Crop Tops Workout Athletic G...,22.99,Mippo,Clothing_Shoes_and_Jewelry
65140,B07SNTJFY4,Ritche Quick Release Leather Watch Band 18mm 2...,19.99,Ritche,Clothing_Shoes_and_Jewelry


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125873 entries, 0 to 125872
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   parent_asin  125873 non-null  object
 1   title        125869 non-null  object
 2   price        83793 non-null   object
 3   store        122659 non-null  object
 4   category     125873 non-null  object
dtypes: object(5)
memory usage: 4.8+ MB


## Map user/item IDs to integers

In [59]:
# As RecBole expects integer IDs, we need to create mappings from the original string IDs to integers.
user_ids: set[str] = set(df_reviews["user_id"])
item_ids: set[str] = set(df_reviews["parent_asin"])

user_map: dict[str, int] = {uid: i+1 for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i+1 for i, pid in enumerate(sorted(item_ids))}
print(f"Users: {len(user_map):,}  Items: {len(item_map):,}")

Users: 23,444  Items: 125,873


In [60]:
df_reviews['uid'] = df_reviews['user_id'].map(user_map)
df_reviews['iid'] = df_reviews['parent_asin'].map(item_map)

## Split train/valid/test with cutoff timestamps

In [61]:
# Only reviews from the target category
df_target_reviews = df_reviews[df_reviews["category"] == TARGET_CATEGORY]
df_target_reviews

,user_id,parent_asin,rating,timestamp,category,uid,iid
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B00Z03RC80,1.0,1616743454733,Beauty_and_Personal_Care,9106,6785
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B085PRT2MP,1.0,1614915977684,Beauty_and_Personal_Care,9106,49592
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B08G81QQ9L,5.0,1612052493701,Beauty_and_Personal_Care,9106,60650
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07YYG76X1,1.0,1609700981786,Beauty_and_Personal_Care,9106,41256
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07X4FKLNK,3.0,1581313195358,Beauty_and_Personal_Care,9106,38269
...,...,...,...,...,...,...,...
99995,AEREAO3HIB2ECGFALOQSQ32VFI7A,B07K2XKD4J,5.0,1645192255435,Beauty_and_Personal_Care,4326,25406
99996,AEREAO3HIB2ECGFALOQSQ32VFI7A,B09X9LL2H8,5.0,1643035429396,Beauty_and_Personal_Care,4326,101315
99997,AEREAO3HIB2ECGFALOQSQ32VFI7A,B09GWLJPTH,4.0,1642441471744,Beauty_and_Personal_Care,4326,89257
99998,AEREAO3HIB2ECGFALOQSQ32VFI7A,B00GMOXGPE,5.0,1631534125032,Beauty_and_Personal_Care,4326,4346


In [62]:
# Determine cutoff timestamps for train/valid/test splits based on the target category only
timestamps = sorted(df_target_reviews["timestamp"].values)
train_end_ts = timestamps[int(len(timestamps) * TRAIN_RATIO)]
valid_end_ts = timestamps[int(len(timestamps) * (TRAIN_RATIO + VALID_RATIO))]
print(f"Train end timestamp: {train_end_ts} ({pd.Timestamp(train_end_ts, unit='ms', tz='UTC')})")
print(f"Valid end timestamp: {valid_end_ts} ({pd.Timestamp(valid_end_ts, unit='ms', tz='UTC')})")

Train end timestamp: 1654641897128 (2022-06-07 22:44:57.128000+00:00)
Valid end timestamp: 1664376740827 (2022-09-28 14:52:20.827000+00:00)


In [63]:
df_reviews_train = df_reviews[df_reviews["timestamp"] <= train_end_ts]
df_reviews_valid = df_reviews[(df_reviews["timestamp"] > train_end_ts) & (df_reviews["timestamp"] <= valid_end_ts)]
df_reviews_test = df_reviews[df_reviews["timestamp"] > valid_end_ts]
print(f"Train reviews: {len(df_reviews_train)}")
print(f"Valid reviews: {len(df_reviews_valid)}")
print(f"Test reviews: {len(df_reviews_test)}")

Train reviews: 161967
Valid reviews: 19407
Test reviews: 18626


In [64]:
df_target_reviews_train = df_target_reviews[df_target_reviews["timestamp"] <= train_end_ts]
df_target_reviews_valid = df_target_reviews[(df_target_reviews["timestamp"] > train_end_ts) & (df_target_reviews["timestamp"] <= valid_end_ts)]
df_target_reviews_test = df_target_reviews[df_target_reviews["timestamp"] > valid_end_ts]
print(f"Target category train reviews: {len(df_target_reviews_train)}")
print(f"Target category valid reviews: {len(df_target_reviews_valid)}")
print(f"Target category test reviews: {len(df_target_reviews_test)}")

Target category train reviews: 80001
Target category valid reviews: 10000
Target category test reviews: 9999


## Define cold vs. warm users with train data in target category

In [65]:
train_counts = df_target_reviews_train.groupby("uid", sort=False).size().rename("num_train")
valid_counts = df_target_reviews_valid.groupby("uid", sort=False).size().rename("num_valid")
test_counts = df_target_reviews_test.groupby("uid", sort=False).size().rename("num_test")

df_user_target_counts = (
    pd.concat([train_counts, valid_counts, test_counts], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
)

df_train_users = df_user_target_counts[df_user_target_counts["num_train"] > 0]
cold_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] < WARM_USER_MIN_REVIEWS]["uid"])
warm_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] >= WARM_USER_MIN_REVIEWS]["uid"])
print(f"Warm users (>= {WARM_USER_MIN_REVIEWS} reviews): {len(warm_user_ids)}")
print(f"Cold users (< {WARM_USER_MIN_REVIEWS} reviews): {len(cold_user_ids)}")

Warm users (>= 5 reviews): 2655
Cold users (< 5 reviews): 12901


## Define Target Items

In [75]:
df_items['is_target'] = (df_items['category'] == TARGET_CATEGORY)
df_items['iid'] = df_items['parent_asin'].map(item_map)
target_item_ids = set(df_items[df_items['is_target']]['iid'])
df_items

,parent_asin,title,price,store,category,is_target,iid
0,B01DX1OEFO,"L.A. COLORS 5 Color Matte Eyeshadow, Brown Twe...",2.49,L.A. COLORS,Beauty_and_Personal_Care,True,8780
1,B0BVYFF3ML,Ayana Marley Hair 6 Packs Marley Twist Braidin...,34.99,Ayana,Beauty_and_Personal_Care,True,119498
2,B09WHMND5W,Newcally Natural Look Lashes Mink 3D Fluffy Wi...,9.45,Newcally,Beauty_and_Personal_Care,True,100674
3,B01M67WKPZ,BLUE LIZARD Australian Sunscreen Blue Lizard S...,27.8,BLUE LIZARD,Beauty_and_Personal_Care,True,10784
4,B07PN759Z4,Shea Butter Clean Hand Wash by South of France...,22.69,South of France Natural Body Care,Beauty_and_Personal_Care,True,29684
...,...,...,...,...,...,...,...
125868,B0BDKYCYRK,Frye Women's Asher Blue Light Glasses Oval Sun...,43.77,Frye,Clothing_Shoes_and_Jewelry,False,111964
125869,B0CCT2CDL6,HUGO Men's Standard Square Logo Swim Trunk,53.48,HUGO,Clothing_Shoes_and_Jewelry,False,125597
125870,B0B8VTSJTJ,Bukesiyi Sasquatch Hat Bigfoot Embroidered Tru...,16.99,Bukesiyi,Clothing_Shoes_and_Jewelry,False,109816
125871,B0C4TC78JH,4 Pairs Bohemian Vintage Dangle Earrings Retro...,9.99,meekoo,Clothing_Shoes_and_Jewelry,False,123225


## Generate user-item history for sequential recommendation

In [76]:
df_sorted: pd.DataFrame = pd.concat(
    [
        df_reviews_train.assign(split="train"),
        df_reviews_valid.assign(split="valid"),
        df_reviews_test.assign(split="test"),
    ],
    ignore_index=True,
).sort_values(
    ["uid", "timestamp"],
    ascending=[True, True],
    kind="mergesort",
)

def build_history(series: pd.Series) -> pd.Series:
    result: list[str] = []
    history: list[int] = []
    for val in series:
        result.append(" ".join(map(str, history[-MAX_ITEM_LIST_LENGTH:])))
        history.append(val)
    return pd.Series(result, index=series.index, dtype='str')

# History for all items (not just target category)
df_sorted["iid_list"] = (
    df_sorted.groupby("uid", sort=False)["iid"]
    .transform(build_history)
)

# History for items in the target category only
df_sorted["target_iid_list"] = ""
target_mask = df_sorted['iid'].isin(target_item_ids)
df_sorted.loc[target_mask, "target_iid_list"] = (
    df_sorted[target_mask]
    .groupby("uid", sort=False)["iid"]
    .transform(build_history)
)

df_sorted

,user_id,parent_asin,rating,timestamp,category,uid,iid,split,iid_list,target_iid_list
155438,AE223GHNZEI5MRMBVVRGJONDNWRQ,B0BQLK5XYR,5.0,1618693968677,Clothing_Shoes_and_Jewelry,1,117609,train,,
155437,AE223GHNZEI5MRMBVVRGJONDNWRQ,B0BML679MG,1.0,1622135989498,Clothing_Shoes_and_Jewelry,1,116254,train,117609,
155436,AE223GHNZEI5MRMBVVRGJONDNWRQ,B07DDK42RF,5.0,1643872499567,Clothing_Shoes_and_Jewelry,1,20917,train,117609 116254,
180538,AE223GHNZEI5MRMBVVRGJONDNWRQ,B01GO0VDQO,5.0,1659421408552,Clothing_Shoes_and_Jewelry,1,9311,valid,117609 116254 20917,
180537,AE223GHNZEI5MRMBVVRGJONDNWRQ,B07WP1VDG9,5.0,1659421641899,Clothing_Shoes_and_Jewelry,1,37553,valid,117609 116254 20917 9311,
...,...,...,...,...,...,...,...,...,...,...
24042,AHZZYA6SBA7TJ6HCR563B5VY7LYA,B0C57HTLY8,5.0,1601850554916,Beauty_and_Personal_Care,23443,123472,train,34559 108633,34559
24041,AHZZYA6SBA7TJ6HCR563B5VY7LYA,B07VBTRNQK,2.0,1601932624696,Beauty_and_Personal_Care,23443,35671,train,34559 108633 123472,34559 123472
118708,AHZZYA6SBA7TJ6HCR563B5VY7LYA,B00NOU3VTK,5.0,1604277455272,Clothing_Shoes_and_Jewelry,23443,5553,train,34559 108633 123472 35671,
24040,AHZZYA6SBA7TJ6HCR563B5VY7LYA,B0BT1BS4DN,5.0,1607801512396,Beauty_and_Personal_Care,23443,118637,train,34559 108633 123472 35671 5553,34559 123472 35671


## Write atomic files

In [77]:
def get_user_category(uid: str) -> int:
    if uid in warm_user_ids:
        return 0 # Warm user
    elif uid in cold_user_ids:
        return 1 # Cold user
    else:
        return 2 # New user
    
def write_user_file(path: Path, uids: set[str]) -> None:
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "user_id:token\t"
            "category:token\n"
        )

        for uid in uids:
            f.write(
                f"{uid}\t"
                f"{get_user_category(uid)}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")

def write_item_file(path: Path, df: pd.DataFrame) -> None:
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "item_id:token\t"
            "store:token\t"
            "price:float\n"
        )

        for _, row in df.iterrows():
            f.write(
                f"{row['iid']}\t"
                f"{row['store'] if pd.notna(row['store']) else ''}\t"
                f"{row['price'] if pd.notna(row['price']) else ''}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")

def write_inter_file(
    path: Path, df: pd.DataFrame,
    item_id_list_field: str
) -> None:
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "user_id:token\t"
            "item_id:token\t"
            "rating:float\t"
            "timestamp:float\t"
            "item_id_list:token_seq\n"
        )

        for _, row in df.iterrows():
            f.write(
                f"{row['uid']}\t"
                f"{row['iid']}\t"
                f"{row['rating']}\t"
                f"{row['timestamp']}\t"
                f"{row[item_id_list_field]}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")


In [78]:
df_target = df_sorted[df_sorted['iid'].isin(target_item_ids)]
df_target_train = df_target[df_target['split'] == 'train']
df_target_valid = df_target[df_target['split'] == 'valid']
df_target_test = df_target[df_target['split'] == 'test']

### Target category

In [79]:
# History only for items in the target category
target_dataset_prefix = Path(DATA_DIR) / "target" / "target"
target_dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(target_dataset_prefix.with_suffix(".train.inter"), 
                 df_target_train, item_id_list_field="target_iid_list")
write_inter_file(target_dataset_prefix.with_suffix(".valid.inter"), 
                 df_target_valid, item_id_list_field="target_iid_list")
write_inter_file(target_dataset_prefix.with_suffix(".test.inter"), 
                 df_target_test, item_id_list_field="target_iid_list")
write_user_file(target_dataset_prefix.with_suffix(".user"), df_target['uid'].unique())
write_item_file(target_dataset_prefix.with_suffix(".item"), df_items[df_items["is_target"]])

Wrote ../data/target/target.train.inter (80,001 rows)
Wrote ../data/target/target.valid.inter (10,000 rows)
Wrote ../data/target/target.test.inter (9,999 rows)
Wrote ../data/target/target.user (17,437 rows)
Wrote ../data/target/target.item (54,195 rows)


### Cross category 

In [80]:
# History includes all cross-category items (not just target category)
cross_dataset_prefix = Path(DATA_DIR) / "cross" / "cross"
cross_dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(cross_dataset_prefix.with_suffix(".train.inter"), 
                 df_target_train, item_id_list_field="iid_list")
write_inter_file(cross_dataset_prefix.with_suffix(".valid.inter"), 
                 df_target_valid, item_id_list_field="iid_list")
write_inter_file(cross_dataset_prefix.with_suffix(".test.inter"), 
                 df_target_test, item_id_list_field="iid_list")
write_user_file(cross_dataset_prefix.with_suffix(".user"), df_target['uid'].unique())
write_item_file(cross_dataset_prefix.with_suffix(".item"), df_items)

Wrote ../data/cross/cross.train.inter (80,001 rows)
Wrote ../data/cross/cross.valid.inter (10,000 rows)
Wrote ../data/cross/cross.test.inter (9,999 rows)
Wrote ../data/cross/cross.user (17,437 rows)
Wrote ../data/cross/cross.item (125,873 rows)
